# OFDM 系统可视化教程

本 Notebook 通过 Python 实现一个完整的 OFDM 通信流程，并通过图形展示每一步的数据变化。

系统流程：

Bits → QPSK → Subcarriers → IFFT → Cyclic Prefix → Channel → FFT → Demod

目标：

- 理解 OFDM 每一步在做什么
- 观察数据结构变化
- 通过图示理解信号处理

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 中文字体
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(0)

## 1 参数设置

OFDM 系统关键参数：

|参数|说明|
|---|---|
|N|子载波数量|
|CP|循环前缀长度|
|bits_per_symbol|调制阶数|

这些参数决定系统容量和抗多径能力。

In [ ]:
N = 64
cp_len = 16
bits_per_symbol = 2

num_bits = N * bits_per_symbol

print("比特数量:", num_bits)

## 2 生成输入比特流

通信系统的输入是 **二进制比特流**：

例如：

1011010101...

在 OFDM 中，这些比特会被映射为复数调制符号。

In [ ]:
bits = np.random.randint(0,2,num_bits)

print(bits[:20])

plt.figure(figsize=(10,2))
plt.step(range(len(bits)),bits)

plt.title("输入比特流")
plt.xlabel("Bit Index")
plt.ylabel("Bit Value")

plt.ylim(-0.2,1.2)
plt.show()

## 3 QPSK 调制

每 2 个比特映射为一个复数符号。

|Bits|Symbol|
|---|---|
|00|1+1j|
|01|-1+1j|
|11|-1-1j|
|10|1-1j|

这一步把数字信息转换为模拟信号符号。

In [ ]:
bit_pairs = bits.reshape((-1,2))

mapping = {
    (0,0):1+1j,
    (0,1):-1+1j,
    (1,1):-1-1j,
    (1,0):1-1j
}

symbols = np.array([mapping[tuple(b)] for b in bit_pairs])

print(symbols[:10])

plt.scatter(symbols.real,symbols.imag)

plt.title("QPSK 星座图")
plt.xlabel("I")
plt.ylabel("Q")

plt.grid()

plt.show()

## 4 子载波映射

OFDM 的核心思想：

**把数据分配到多个子载波。**

频域表示：

X[k]

每个子载波承载一个调制符号。

In [ ]:
plt.figure()

plt.stem(np.abs(symbols))

plt.title("OFDM 子载波幅度")
plt.xlabel("Subcarrier index")
plt.ylabel("Amplitude")

plt.show()

## 5 IFFT 生成 OFDM 时域信号

OFDM 调制通过 IFFT 实现：

x[n] = IFFT(X[k])

IFFT 会把多个子载波叠加成一个时域信号。

In [ ]:
time_signal = np.fft.ifft(symbols)

plt.plot(time_signal.real,label="Real")
plt.plot(time_signal.imag,label="Imag")

plt.title("OFDM 时域信号")

plt.legend()

plt.show()

## 6 添加循环前缀 (CP)

循环前缀用于对抗多径传播。

方法：

复制 OFDM 符号尾部到前面。

结构：

|CP| OFDM symbol |

In [ ]:
cp = time_signal[-cp_len:]

tx_signal = np.concatenate([cp,time_signal])

plt.plot(tx_signal.real)

plt.axvline(cp_len,color="r",label="CP boundary")

plt.title("带循环前缀的 OFDM 符号")

plt.legend()

plt.show()

## 7 信道 (AWGN)

真实无线信道会加入噪声。

最常见的模型是 **AWGN 信道**。

In [ ]:
snr_db = 20

signal_power = np.mean(np.abs(tx_signal)**2)
noise_power = signal_power/(10**(snr_db/10))

noise = np.sqrt(noise_power/2)*(np.random.randn(len(tx_signal)) + 1j*np.random.randn(len(tx_signal)))

rx_signal = tx_signal + noise

plt.plot(rx_signal.real)

plt.title("接收信号")

plt.show()

## 8 FFT 恢复子载波

接收端去掉 CP 后进行 FFT：

X[k] = FFT(x[n])

这样可以恢复每个子载波符号。

In [ ]:
rx_no_cp = rx_signal[cp_len:]

rx_symbols = np.fft.fft(rx_no_cp)

plt.scatter(rx_symbols.real,rx_symbols.imag)

plt.title("接收星座图")

plt.grid()

plt.show()

# 总结

OFDM 数据流：

bits
 ↓
QPSK symbols
 ↓
subcarriers
 ↓
IFFT
 ↓
time signal
 ↓
CP
 ↓
channel
 ↓
FFT
 ↓
recovered symbols

通过可视化可以直观理解 OFDM 的信号处理过程。